In [1]:
import pandas as pd

In [2]:
df_Trans = pd.read_parquet('transactions.parquet')
df_TransItem = pd.read_parquet('transaction_items.parquet')
df_Users = pd.read_parquet('users.parquet')
df_stores = pd.read_parquet('stores.parquet')
df_menu = pd.read_parquet('menu_items.parquet')
df_voucher = pd.read_parquet('vouchers.parquet')
df_payment = pd.read_parquet('paymentsUnique.parquet')

## Mensamakan tipe data

In [3]:
df_Trans['transaction_id'] = df_Trans['transaction_id'].astype(str)
df_Trans['user_id'] = (
    pd.to_numeric(df_Trans['user_id'], errors='coerce')
    .astype('Int64')
)
df_Trans['voucher_id']= df_Trans['voucher_id'].astype('Int64')
df_Trans['created_at'] = pd.to_datetime(df_Trans['created_at'], errors='coerce')
df_TransItem['transaction_id'] = df_TransItem['transaction_id'].astype(str)
df_TransItem['created_at'] = pd.to_datetime(df_TransItem['created_at'], errors='coerce')
df_Users['birthdate'] = pd.to_datetime(df_Users['birthdate'], errors='coerce')
df_Users['registered_at'] = pd.to_datetime(df_Users['registered_at'], errors='coerce')
df_Users['user_id'] = (
    pd.to_numeric(df_Users['user_id'], errors='coerce')
    .astype('Int64')
)
df_Users['gender'] = df_Users['gender'].astype(str)
df_menu[['item_name', 'category']] = df_menu[['item_name', 'category']].astype(str)
df_stores[['store_name', 'street', 'city', 'state']] = df_stores[['store_name', 'street', 'city', 'state']].astype(str)
df_stores['postal_code'] = df_stores['postal_code'].astype(object)
df_voucher['valid_from'] = pd.to_datetime(df_voucher['valid_from'])
df_voucher['valid_to'] = pd.to_datetime(df_voucher['valid_to'])


In [4]:
df_Trans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14623691 entries, 0 to 14623690
Data columns (total 9 columns):
 #   Column             Dtype         
---  ------             -----         
 0   transaction_id     object        
 1   store_id           int8          
 2   payment_method_id  int8          
 3   voucher_id         Int64         
 4   user_id            Int64         
 5   original_amount    float32       
 6   discount_applied   float32       
 7   final_amount       float32       
 8   created_at         datetime64[ns]
dtypes: Int64(2), datetime64[ns](1), float32(3), int8(2), object(1)
memory usage: 669.4+ MB


In [5]:
df_TransItem.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29246323 entries, 0 to 29246322
Data columns (total 6 columns):
 #   Column          Dtype         
---  ------          -----         
 0   transaction_id  object        
 1   item_id         int8          
 2   quantity        int8          
 3   unit_price      float32       
 4   subtotal        float32       
 5   created_at      datetime64[ns]
dtypes: datetime64[ns](1), float32(2), int8(2), object(1)
memory usage: 725.2+ MB


In [6]:
df_menu.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   item_id         8 non-null      int8   
 1   item_name       8 non-null      object 
 2   category        8 non-null      object 
 3   price           8 non-null      float32
 4   is_seasonal     8 non-null      bool   
 5   available_from  0 non-null      float32
 6   available_to    0 non-null      float32
dtypes: bool(1), float32(3), int8(1), object(2)
memory usage: 372.0+ bytes


In [7]:
df_Users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2196257 entries, 0 to 2196256
Data columns (total 4 columns):
 #   Column         Dtype         
---  ------         -----         
 0   user_id        Int64         
 1   gender         object        
 2   birthdate      datetime64[ns]
 3   registered_at  datetime64[ns]
dtypes: Int64(1), datetime64[ns](2), object(1)
memory usage: 69.1+ MB


In [8]:
df_stores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   store_id     10 non-null     int8   
 1   store_name   10 non-null     object 
 2   street       10 non-null     object 
 3   postal_code  10 non-null     object 
 4   city         10 non-null     object 
 5   state        10 non-null     object 
 6   latitude     10 non-null     float32
 7   longitude    10 non-null     float32
dtypes: float32(2), int8(1), object(5)
memory usage: 622.0+ bytes


In [9]:
def run_deep_validation(df_trans, df_items, df_users, df_stores):
    print("--- RUNNING DEEP VALIDATION ---")
    
    # A. Primary Key Uniqueness
    pk_check = df_trans['transaction_id'].duplicated().sum()
    print(f"1. Duplicate Transaction IDs: {pk_check}")
    
    # B. Foreign Key Integrity (Check Store IDs)
    invalid_stores = df_trans[~df_trans['store_id'].isin(df_stores['store_id'])]
    print(f"2. Transaksi dengan Store ID tidak terdaftar: {len(invalid_stores)}")
    
    # C. Impossible Dates Check
    today = pd.Timestamp.now()
    future_trans = df_trans[df_trans['created_at'] > today]
    print(f"3. Transaksi dari 'Masa Depan': {len(future_trans)}")
    
    # D. Age Validity (Mathematical Outlier)
    # Menghitung umur saat registrasi
    temp_users = df_users.dropna(subset=['birthdate', 'registered_at'])
    age_at_reg = (temp_users['registered_at'] - temp_users['birthdate']).dt.days / 365.25
    impossible_age = temp_users[(age_at_reg < 12) | (age_at_reg > 100)]
    print(f"4. User dengan umur tidak masuk akal (<12 atau >100): {len(impossible_age)}")
    
    return pk_check, len(invalid_stores), len(future_trans)

run_deep_validation(df_trans=df_Trans, df_items=df_TransItem, df_users=df_Users, df_stores=df_stores)

--- RUNNING DEEP VALIDATION ---
1. Duplicate Transaction IDs: 0
2. Transaksi dengan Store ID tidak terdaftar: 0
3. Transaksi dari 'Masa Depan': 0
4. User dengan umur tidak masuk akal (<12 atau >100): 0


(0, 0, 0)

In [10]:
df_Trans.to_parquet( "transactions.parquet")
df_TransItem.to_parquet( "transaction_items.parquet")
df_Users.to_parquet( "users.parquet")
df_stores.to_parquet( "stores.parquet")
df_menu.to_parquet( "menu_items.parquet")
df_voucher.to_parquet("vouchers.parquet")
df_payment.to_parquet("paymentsUnique.parquet")

In [11]:
import gc
gc.collect()

18